# LC #20 — Valid Parentheses
**Category:** Stack | **Difficulty:** Easy | **Day 3**

---

<div style="padding:15px;border-left:8px solid #f7971e;background:#fff8e1;border-radius:4px;">
<strong>The Problem:</strong> Given string <code>s</code> containing <code>()[]{}</code>,
return <code>true</code> if the string is valid. Valid means every opener has a matching
closer in the correct order, and they are properly nested.
</div>

**Examples:**
```
"()[]{}"   → true
"()[]{}"   → true
"(]"       → false  (wrong closer)
"([)]"     → false  (wrong nesting order)
"{[]}"     → true   (properly nested)
```

**Core Insight:** Push openers onto the stack. For each closer, check that the top of the stack is its matching opener. Any mismatch or leftover items = invalid.

## Mental Models

**1. The Stack as "Pending Openers"**
The stack holds openers that haven't been matched yet. They're "waiting" for their closing partner. LIFO (last in, first out) naturally enforces proper nesting — the most recent opener must be closed first.

**2. The Pairing Map**
`{')': '(', ']': '[', '}': '{'}` maps each closer to its expected opener. This avoids long if-elif chains and makes the logic extensible to new bracket types.

**3. Two Failure Modes**
- Mismatch: closer doesn't match top of stack (wrong type or empty stack)
- Leftover: stack is non-empty at end (unclosed openers)

In [ ]:
# No efficient "brute force" exists — the stack approach IS the optimal.
# This shows what iterative shrinking would look like (O(n²)):

def isValid_shrink(s: str) -> bool:
    """Replace matching pairs iteratively until no more can be removed."""
    prev = None
    while s != prev:
        prev = s
        for pair in ["()", "[]", "{}"]:
            s = s.replace(pair, "")
    return s == ""

# Correct but O(n²) — each pass is O(n), up to n/2 passes
print(isValid_shrink("()[]{}"))   # True
print(isValid_shrink("([)]"))     # False
print(isValid_shrink("{[]}"))     # True

## Step-by-Step: `"([)]"`

| i | char | Action | Stack |
|---|------|--------|-------|
| 0 | `(` | opener → push | `[(]` |
| 1 | `[` | opener → push | `[(, []` |
| 2 | `)` | closer → top is `[`, expected `(` → **MISMATCH** | return False |

The key: at step 2, `)` expects `(` on top but finds `[`. The string is invalid because `[` was opened more recently and must be closed first.

**Correct nesting:** `{[()]}` — innermost opens last, closes first.

In [ ]:
# Optimal — O(n) time, O(n) space
# Stack: push openers, pop and verify for closers.

def isValid(s: str) -> bool:
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}

    for char in s:
        if char in '([{':
            stack.append(char)
        elif char in ')]}':            if not stack or stack[-1] != pairs[char]:
                return False
            stack.pop()

    return len(stack) == 0   # empty = all openers matched

# Test
print(isValid("()[]{}"))   # True
print(isValid("(]"))       # False
print(isValid("([)]"))     # False
print(isValid("{[]}"))     # True
print(isValid(""))         # True (empty string is valid)
print(isValid("("))        # False (unclosed opener)

## Complexity Analysis

| Approach | Time | Space |
|----------|------|-------|
| Iterative shrink | O(n²) | O(n) |
| **Stack (Optimal)** | **O(n)** | **O(n)** |

**Space note:** In the worst case (all openers, no closers), the stack holds n items: `((((...` → O(n) space.

**Best case:** Alternating opener-closer: `()()()` → stack never exceeds size 1.

## Interview Q&A

**Q: Why check `not stack` before `stack[-1]`?**
A: Short-circuit evaluation. If the stack is empty and we see a closer, it's immediately invalid — there's nothing to match. Checking `not stack` first prevents an IndexError on `stack[-1]`.

**Q: What happens with an empty string?**
A: The loop never runs, stack remains empty, `len(stack) == 0` returns True. An empty string is considered valid.

**Q: Real-world applications?**
A: Validating nested structures: JSON/XML parsing, nested SQL subqueries, template engines (Jinja, Handlebars), config file validation (YAML, TOML). At Citi: validating nested alert condition expressions.

**Q: How would you extend this for additional bracket types?**
A: Just add entries to the `pairs` dict. The algorithm generalizes to any matching delimiter pair without code changes — a good extensibility answer.

## The Citi Angle

**Nested configuration validation.** At Citi, monitoring rules were expressed as nested conditions:

```python
# Validate nested alert condition expression
def validate_condition(expr: str) -> bool:
    """
    Valid: "(cpu > 80 AND (mem > 70 OR disk > 90))"
    Invalid: "(cpu > 80 AND [mem > 70])"  — mixed brackets
    """
    stack = []
    pairs = {')': '(', ']': '[', '}': '{'}
    for char in expr:
        if char in '([{':
            stack.append(char)
        elif char in ')]}':            if not stack or stack[-1] != pairs[char]:
                return False
            stack.pop()
    return len(stack) == 0

tests = [
    ("(cpu > 80 AND (mem > 70 OR disk > 90))", True),
    ("(cpu > 80 AND [mem > 70])", False),
]
for expr, expected in tests:
    result = validate_condition(expr)
    status = "PASS" if result == expected else "FAIL"
    print(f"{status}: {repr(expr[:40])} → {result}")
```

**Interview tie-in:** "Valid parentheses is the foundation of any parser or config validator. I used this pattern to validate nested monitoring expressions — ensuring alert conditions were syntactically correct before deployment."

## Summary

| | Value |
|--|--|
| **Pattern** | Stack — push openers, pop and verify for closers |
| **Time** | O(n) |
| **Space** | O(n) |
| **Key insight** | LIFO naturally enforces proper nesting |
| **Say in interview** | "Push openers. For each closer, top of stack must be the matching opener. Empty stack at end = valid." |

**Two failure modes to mention:** Wrong closer type (mismatch). Unclosed openers remaining in stack (leftover).